<a href="https://colab.research.google.com/github/SiyumiJayawardhane/freshsense-imagemodel/blob/siyumi/FreshSenseYoloV5ImageModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. Mount Google Drive**

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**2. Define paths and create the new YOLO dataset structure**

In [5]:
import os
import shutil
import random
from pathlib import Path

BASE_DRIVE_PATH = "/content/drive/MyDrive/FreshSense/Dataset"

# New unified dataset folder
DATASET_ROOT = "/content/food_spoilage_dataset"
os.makedirs(f"{DATASET_ROOT}/images/train", exist_ok=True)
os.makedirs(f"{DATASET_ROOT}/images/val", exist_ok=True)
os.makedirs(f"{DATASET_ROOT}/labels/train", exist_ok=True)
os.makedirs(f"{DATASET_ROOT}/labels/val", exist_ok=True)

print("Folder structure created")

Folder structure created


**3. Reorganize + Remap class IDs**

In [6]:
# Global 9-class list (order matters!)
class_names = [
    "fresh_banana", "atrisk_banana", "spoiled_banana",
    "fresh_cucumber", "atrisk_cucumber", "spoiled_cucumber",
    "fresh_tomato", "atrisk_tomato", "spoiled_tomato"
]

# Mapping: original class_id (per item) → global class_id
banana_mapping   = {0: 1, 1: 0, 2: 2}   # atrisk=0→1, fresh=1→0, spoiled=2→2
cucumber_mapping = {0: 4, 1: 3, 2: 5}
tomato_mapping   = {0: 7, 1: 6, 2: 8}

datasets = {
    "banana_dataset":   ("banana",   banana_mapping),
    "cucumber_dataset": ("cucumber", cucumber_mapping),
    "tomato_dataset":   ("tomato",   tomato_mapping)
}

all_image_label_pairs = []   # will store (image_path, label_path, new_class_id)

for dataset_name, (item_name, mapping) in datasets.items():
    dataset_path = os.path.join(BASE_DRIVE_PATH, dataset_name)
    if not os.path.exists(dataset_path):
        print(f" {dataset_name} not found")
        continue

    # The three class folders inside each dataset
    class_folders = ["atrisk", "fresh", "spoiled"] if item_name != "banana" else ["atrisk_banana", "fresh_banana", "spoiled_banana"]

    for idx, folder in enumerate(class_folders):
        class_dir = os.path.join(dataset_path, folder)
        if not os.path.exists(class_dir):
            print(f"Missing folder: {class_dir}")
            continue

        images_dir = os.path.join(class_dir, "images")
        labels_dir = os.path.join(class_dir, "labels")

        if not os.path.exists(images_dir) or not os.path.exists(labels_dir):
            continue

        original_class_id = idx   # 0,1,2 based on folder order
        global_class_id = mapping[original_class_id]

        for img_file in os.listdir(images_dir):
            if not img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue

            base_name = os.path.splitext(img_file)[0]
            label_file = base_name + ".txt"
            label_path = os.path.join(labels_dir, label_file)

            if os.path.exists(label_path):
                all_image_label_pairs.append((os.path.join(images_dir, img_file), label_path, global_class_id))

print(f"Found {len(all_image_label_pairs)} labeled images ready for merging")

Found 3201 labeled images ready for merging
